# Panel-data pilot: FLUX.2 klein 4B on Kaggle T4

Pilot for a 2-4 panel software-concept image generator. Questions: (1) does klein 4B run stably in fp16 on a T4,
(2) real images/hour at 512x512 and 4 steps per T4 and with both, (3) does passing panel 1 as a reference keep panels 2-4 consistent.
Needs: GPU T4 x2 and Internet on. Outputs land in /kaggle/working/out (PNGs, contact_sheet.html, results.json, panels.zip).

In [ ]:
import subprocess, sys
print(subprocess.run("pip install -q -U git+https://github.com/huggingface/diffusers transformers accelerate sentencepiece huggingface_hub 2>&1 | tail -3", shell=True, capture_output=True, text=True).stdout)
import diffusers, transformers, torch
print("diffusers", diffusers.__version__, "transformers", transformers.__version__, "torch", torch.__version__)

In [ ]:
import subprocess, torch
print(subprocess.run("nvidia-smi", shell=True, capture_output=True, text=True).stdout)
N_GPU = torch.cuda.device_count()
GPU_INFO = []
for i in range(N_GPU):
    p = torch.cuda.get_device_properties(i)
    GPU_INFO.append({"index": i, "name": p.name, "total_gb": round(p.total_memory / 1e9, 2), "capability": f"{p.major}.{p.minor}", "bf16_native": torch.cuda.is_bf16_supported(including_emulation=False) if i == 0 else None})
    print(GPU_INFO[-1])
import os
os.makedirs("/kaggle/working/out", exist_ok=True)
print("RAM:", subprocess.run("free -g | head -2", shell=True, capture_output=True, text=True).stdout)
print("disk:", subprocess.run("df -h /kaggle/working /root | tail -2", shell=True, capture_output=True, text=True).stdout)

In [ ]:
# Concept scripts (same content as scripts/concepts_pilot.json)
CONCEPTS_JSON = r'''{
  "model": "black-forest-labs/FLUX.2-klein-4B",
  "style_prefix": "Flat vector illustration, simple shapes, limited palette, white background, no text, no letters.",
  "reference_instruction": "Keep exactly the same visual style, colour palette and objects as the reference image, only change the moment shown.",
  "width": 512,
  "height": 512,
  "steps": 4,
  "mode_a_concepts": ["sandbox", "config", "cache", "queue", "ratelimit", "depgraph"],
  "concepts": [
    {"id": "sandbox", "title": "Sandbox isolation", "seed": 1001, "panels": [
      "a small orange cube (the code) arrives from the left side of the frame",
      "a blue box closes around the orange cube like a lid coming down",
      "the orange cube reaches an arm outward through the blue box and is stopped by a grey barrier",
      "the orange cube rests safely inside the closed blue box, a soft green glow around it"]},
    {"id": "config", "title": "Config loader with schema validation", "seed": 1002, "panels": [
      "a small orange file (the config) slides in from the left toward a blue gate",
      "the orange file is placed against a blue stencil shaped like a template to compare its shape",
      "one corner of the orange file does not fit the blue stencil and a red mark appears on it",
      "the orange file has a corrected corner, fits the blue stencil exactly and a green glow appears"]},
    {"id": "logging", "title": "Centralized logging", "seed": 1003, "panels": [
      "three small blue circles (services) stand in a row at the left, each holding a small orange note",
      "each blue circle sends its orange note along a thin line toward a large grey funnel in the centre",
      "the orange notes pile up inside the large grey funnel in neat order",
      "a large blue shelf at the right holds the orange notes in a tidy stack, one note highlighted by a green ring"]},
    {"id": "cache", "title": "Caching", "seed": 1004, "panels": [
      "a small blue person-shaped circle (the client) asks a far away large grey cylinder (the database) for an orange cube, a long thin line between them",
      "the orange cube travels back along the long line and passes a small yellow box (the cache) sitting near the client",
      "the orange cube is stored in the small yellow box next to the blue client circle",
      "the blue client circle asks again and gets the orange cube instantly from the small yellow box, a very short line between them"]},
    {"id": "retry", "title": "Retries with backoff", "seed": 1005, "panels": [
      "a small blue circle (the client) throws an orange ball at a grey wall (the server) and it bounces back",
      "the blue circle waits a short moment, shown by one small grey dot, then throws the orange ball again and it bounces back again",
      "the blue circle waits a longer moment, shown by three small grey dots, and throws the orange ball a third time",
      "the orange ball passes through an open gap in the grey wall and a green glow appears"]},
    {"id": "queue", "title": "A queue with workers", "seed": 1006, "panels": [
      "many small orange cubes (the jobs) arrive from the left and line up in a row on a grey conveyor belt",
      "the row of orange cubes moves along the grey conveyor belt toward two blue robot arms (the workers)",
      "each blue robot arm picks up one orange cube from the front of the row",
      "the blue robot arms place finished green cubes on a shelf at the right, the row of orange cubes is shorter"]},
    {"id": "auth", "title": "Authentication tokens", "seed": 1007, "panels": [
      "a small blue circle (the user) shows a small grey key to a blue door guard at a gate",
      "the blue door guard hands the blue circle a small round orange token in exchange for the key",
      "the blue circle holds the orange token up at a second gate to a room with a grey cylinder (the data)",
      "the gate opens and the blue circle walks through with the orange token, the grey cylinder is in reach"]},
    {"id": "migration", "title": "A database migration", "seed": 1008, "panels": [
      "a large grey cylinder (the old database) holds a stack of orange rows with two columns",
      "a blue crane lifts a new blue column shape and hovers it above the stack of orange rows",
      "the blue column is slotted into every orange row of the stack, the rows are now three columns wide",
      "the stack of orange rows with three columns sits back in the grey cylinder, a green glow around it"]},
    {"id": "ratelimit", "title": "Rate limiting", "seed": 1009, "panels": [
      "a crowd of small orange dots (requests) rushes from the left toward a blue gate",
      "the blue gate lets a few orange dots through one at a time while the others bunch up behind it",
      "the other orange dots are held back by a grey bar across the gate, a small dial on the gate points high",
      "the grey bar lifts again and orange dots flow through steadily in a thin line to the server on the right"]},
    {"id": "ci", "title": "CI build pipeline", "seed": 1010, "panels": [
      "a small orange cube (the commit) sits at the left start of a long grey pipe made of three sections",
      "the orange cube travels into the first section of the grey pipe, which glows blue, as it builds",
      "the orange cube passes into the second section of the pipe, which glows blue, as it is tested",
      "the orange cube exits the end of the pipe as a green cube on a shelf at the right, a green glow around it"]},
    {"id": "depgraph", "title": "Dependency graph", "seed": 1011, "panels": [
      "one large orange circle (the app) sits alone in the centre of the frame",
      "two blue circles connect to the orange circle by thin lines, the app needs them",
      "each of the two blue circles connects by thin lines to one or two smaller grey circles",
      "the whole shape is a small tree of connected circles, the orange circle at the top, the grey circles at the bottom, all lines pointing downward"]},
    {"id": "merge", "title": "A diff and merge", "seed": 1012, "panels": [
      "two stacks of coloured bars, one orange and one blue, stand side by side, each stack nearly the same as the other",
      "one bar in the orange stack and one bar in the blue stack are highlighted with a yellow glow, the rest are dimmed",
      "an arrow carries the highlighted blue bar over the orange stack to replace its highlighted bar",
      "a single combined stack of bars stands in the middle with a green glow, the two original stacks are faded behind it"]}
  ],
  "style_tests": [
    {"id": "styletest1", "title": "Style test: server and padlock", "seed": 2001, "prompt": "a grey server box with a blue padlock on its front and a small orange cube beside it"},
    {"id": "styletest2", "title": "Style test: branching tree", "seed": 2002, "prompt": "a branching tree of blue circles joined by thin grey lines, one orange circle at the root"}
  ]
}
'''
import json
spec = json.loads(CONCEPTS_JSON)
print(len(spec['concepts']), 'concepts,', len(spec['style_tests']), 'style tests')

In [ ]:
# Download weights once (so the two workers do not race on the cache) and time it.
import os, time
from huggingface_hub import snapshot_download
t0 = time.time()
try:
    path = snapshot_download(spec["model"])
except Exception as e:
    print("DOWNLOAD FAILED:", repr(e))
    raise
DOWNLOAD_S = round(time.time() - t0, 1)
print("downloaded to", path, "in", DOWNLOAD_S, "s")
print(subprocess.run(f"du -sh {path}/*", shell=True, capture_output=True, text=True).stdout)

In [ ]:
%%writefile /kaggle/working/worker.py
"""One FLUX.2 klein 4B worker, pinned to one GPU by CUDA_VISIBLE_DEVICES.

Usage: python worker.py <spec.json> <out_dir> <result.json>

Stage 1 loads only the text encoder, encodes every prompt this worker needs, then frees it.
Stage 2 loads only the transformer + VAE. This keeps a T4 (15 GB) inside memory without CPU offload.
Dtype ladder if an image comes out black/NaN: fp16 -> fp16 + fp32 VAE -> bf16 + fp32 VAE.
"""
import gc
import json
import os
import shutil
import sys
import time

import numpy as np
import torch
from PIL import Image

spec = json.load(open(sys.argv[1]))
OUT = sys.argv[2]
RESULT = sys.argv[3]
M = spec["model"]
W, H, STEPS = spec["width"], spec["height"], spec["steps"]
jobs = spec["jobs"]
LADDER = [("fp16", torch.float16, False), ("fp16+vae32", torch.float16, True), ("bf16+vae32", torch.bfloat16, True)]

from diffusers import Flux2KleinPipeline  # noqa: E402

res = {"gpu": torch.cuda.get_device_name(0), "gpu_total_gb": round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2),
       "worker": spec["worker"], "load": {}, "images": [], "events": []}


def log(*a):
    print(f"[w{spec['worker']}]", *a, flush=True)


def save_result():
    json.dump(res, open(RESULT, "w"), indent=1)


def gb(x):
    return round(x / 1e9, 2)


def free():
    gc.collect()
    torch.cuda.empty_cache()


def bad(img):
    a = np.asarray(img).astype(np.float32)
    if not np.isfinite(a).all():
        return True
    return bool(a.mean() < 5 or a.std() < 2)


# ---- stage 1: text embeddings (computed once per unique prompt)
prompts = []
for j in jobs:
    if j["prompt"] not in prompts:
        prompts.append(j["prompt"])


def encode_all(dtype):
    t0 = time.perf_counter()
    pipe = Flux2KleinPipeline.from_pretrained(M, transformer=None, vae=None, dtype=dtype)
    pipe.text_encoder.to("cuda")
    torch.cuda.synchronize()
    res["load"]["text_encoder_load_s"] = round(time.perf_counter() - t0, 1)
    torch.cuda.reset_peak_memory_stats()
    t1 = time.perf_counter()
    embeds = {}
    with torch.no_grad():
        for p in prompts:
            e, _ = pipe.encode_prompt(prompt=p, device="cuda")
            embeds[p] = e.cpu()
    torch.cuda.synchronize()
    res["load"]["text_encode_s"] = round(time.perf_counter() - t1, 1)
    res["load"]["text_encode_n_prompts"] = len(prompts)
    res["load"]["text_stage_peak_gb"] = gb(torch.cuda.max_memory_allocated())
    res["load"]["text_dtype"] = str(dtype)
    del pipe
    free()
    return embeds


embeds = encode_all(torch.float16)
if any((not torch.isfinite(e).all()) for e in embeds.values()):
    res["events"].append("fp16 text embeddings contained NaN/inf; re-encoding text in bf16")
    log(res["events"][-1])
    embeds = encode_all(torch.bfloat16)
log("text stage done", res["load"])


# ---- stage 2: transformer + VAE
def load_pipe(level):
    name, dtype, vae32 = LADDER[level]
    t0 = time.perf_counter()
    pipe = Flux2KleinPipeline.from_pretrained(M, text_encoder=None, tokenizer=None, dtype=dtype)
    pipe.to("cuda")
    if vae32:
        pipe.vae.to(torch.float32)
    pipe.set_progress_bar_config(disable=True)
    torch.cuda.synchronize()
    res["load"][f"diffusion_load_s_{name}"] = round(time.perf_counter() - t0, 1)
    log("loaded", name, res["load"][f"diffusion_load_s_{name}"], "s; vram", gb(torch.cuda.memory_allocated()))
    return pipe


def generate(pipe, level, job, ref):
    name, dtype, vae32 = LADDER[level]
    g = torch.Generator(device="cuda").manual_seed(job["seed"])
    emb = embeds[job["prompt"]].to("cuda", dtype=dtype)
    kw = dict(prompt=None, prompt_embeds=emb, height=H, width=W, num_inference_steps=STEPS, guidance_scale=1.0, generator=g)
    if ref is not None:
        kw["image"] = ref
    torch.cuda.synchronize()
    t0 = time.perf_counter()
    with torch.no_grad():
        if vae32:
            lat = pipe(output_type="latent", **kw).images
            x = pipe.vae.decode(lat.to(torch.float32), return_dict=False)[0]
            img = pipe.image_processor.postprocess(x, output_type="pil")[0]
        else:
            img = pipe(**kw).images[0]
    torch.cuda.synchronize()
    return img, time.perf_counter() - t0


level = 0
pipe = load_pipe(level)
torch.cuda.reset_peak_memory_stats()
made = {}  # (concept, mode, panel) -> PIL
first = True
for job in jobs:
    key = (job["concept"], job["mode"], job["panel"])
    if job.get("copy_from"):
        src = tuple(job["copy_from"])
        made[key] = made[src]
        made[key].save(os.path.join(OUT, job["file"]))
        res["images"].append({"file": job["file"], "concept": job["concept"], "mode": job["mode"], "panel": job["panel"],
                              "copied_from": "_".join(map(str, src)), "seconds": 0.0, "counted": False})
        continue
    ref = made[tuple(job["ref"])] if job.get("ref") else None
    attempts = []
    while True:
        try:
            img, secs = generate(pipe, level, job, ref)
        except torch.cuda.OutOfMemoryError as e:
            res["events"].append(f"OOM on {job['file']}: {e}")
            log(res["events"][-1])
            raise
        isbad = bad(img)
        a = np.asarray(img).astype(np.float32)
        attempts.append({"dtype": LADDER[level][0], "seconds": round(secs, 3), "bad": isbad, "mean": round(float(a.mean()), 2), "std": round(float(a.std()), 2)})
        if not isbad or level == len(LADDER) - 1:
            break
        res["events"].append(f"{job['file']} was black/NaN in {LADDER[level][0]} (mean {a.mean():.2f}, std {a.std():.2f}); escalating")
        log(res["events"][-1])
        level += 1
        del pipe
        free()
        pipe = load_pipe(level)
    img.save(os.path.join(OUT, job["file"]))
    made[key] = img
    last = attempts[-1]
    res["images"].append({"file": job["file"], "concept": job["concept"], "mode": job["mode"], "panel": job["panel"],
                          "ref": bool(ref is not None), "dtype": last["dtype"], "seconds": last["seconds"], "bad": last["bad"],
                          "mean": last["mean"], "std": last["std"], "attempts": attempts, "first_image": first,
                          "counted": True, "peak_vram_gb": gb(torch.cuda.max_memory_allocated()),
                          "peak_reserved_gb": gb(torch.cuda.max_memory_reserved())})
    first = False
    log(f"{job['file']} {last['seconds']:.2f}s {last['dtype']} bad={last['bad']} mean={last['mean']}")
    save_result()

res["peak_vram_gb"] = gb(torch.cuda.max_memory_allocated())
res["peak_reserved_gb"] = gb(torch.cuda.max_memory_reserved())
res["final_dtype"] = LADDER[level][0]
res["done"] = True
save_result()
log("done")

In [ ]:
# Build the job list and launch one worker per GPU (CUDA_VISIBLE_DEVICES pins each). Workers run concurrently.
import json, os, signal, subprocess, sys, time
OUT = "/kaggle/working/out"
STYLE, REFI = spec["style_prefix"], spec["reference_instruction"]
mode_a = set(spec["mode_a_concepts"])
n_workers = max(1, N_GPU)
worker_jobs = [[] for _ in range(n_workers)]
for ci, c in enumerate(spec["concepts"]):
    js = worker_jobs[ci % n_workers]
    cid, seed = c["id"], c["seed"]
    js.append(dict(concept=cid, mode="b", panel=1, seed=seed, prompt=f"{STYLE} {c['panels'][0]}", file=f"{cid}_b_1.png"))
    if cid in mode_a:
        js.append(dict(concept=cid, mode="a", panel=1, seed=seed, prompt=js[-1]["prompt"], file=f"{cid}_a_1.png", copy_from=[cid, "b", 1]))
    for p in range(2, 5):
        js.append(dict(concept=cid, mode="b", panel=p, seed=seed, prompt=f"{STYLE} {REFI} {c['panels'][p-1]}", file=f"{cid}_b_{p}.png", ref=[cid, "b", 1]))
    if cid in mode_a:
        for p in range(2, 5):
            js.append(dict(concept=cid, mode="a", panel=p, seed=seed, prompt=f"{STYLE} {c['panels'][p-1]}", file=f"{cid}_a_{p}.png"))
for si, s in enumerate(spec["style_tests"]):
    worker_jobs[si % n_workers].append(dict(concept=s["id"], mode="single", panel=1, seed=s["seed"], prompt=f"{STYLE} {s['prompt']}", file=f"{s['id']}_single_1.png"))
print("jobs per worker:", [sum(1 for j in w if not j.get('copy_from')) for w in worker_jobs], "(generated images; copies of panel 1 excluded)")

procs = []
t_launch = time.time()
for w in range(n_workers):
    sp = {k: spec[k] for k in ("model", "width", "height", "steps")}
    sp.update(worker=w, jobs=worker_jobs[w])
    json.dump(sp, open(f"/kaggle/working/spec_w{w}.json", "w"))
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(w) if N_GPU else "")
    log = open(f"/kaggle/working/worker_{w}.log", "w")
    procs.append(subprocess.Popen([sys.executable, "/kaggle/working/worker.py", f"/kaggle/working/spec_w{w}.json", OUT, f"/kaggle/working/result_w{w}.json"],
                                  stdout=log, stderr=subprocess.STDOUT, env=env, start_new_session=True))
    print("launched worker", w, "pid", procs[-1].pid)

def kill_all():
    for p in procs:
        if p.poll() is None:
            try: os.killpg(p.pid, signal.SIGTERM)
            except ProcessLookupError: pass
    time.sleep(3)
    for p in procs:
        if p.poll() is None:
            try: os.killpg(p.pid, signal.SIGKILL)
            except ProcessLookupError: pass

seen = [0] * n_workers
try:
    while any(p.poll() is None for p in procs):
        time.sleep(10)
        for w in range(n_workers):
            lines = open(f"/kaggle/working/worker_{w}.log").read().splitlines()
            for l in lines[seen[w]:]:
                print(l[:300])
            seen[w] = len(lines)
        if time.time() - t_launch > 4 * 3600:
            print("TIMEOUT"); break
finally:
    kill_all()
WALL_S = round(time.time() - t_launch, 1)
for w in range(n_workers):
    for l in open(f"/kaggle/working/worker_{w}.log").read().splitlines()[seen[w]:]:
        print(l[:300])
print("worker exit codes:", [p.returncode for p in procs], "wall", WALL_S, "s")

In [ ]:
# results.json
import json, os, statistics
workers = []
for w in range(n_workers):
    f = f"/kaggle/working/result_w{w}.json"
    if os.path.exists(f):
        workers.append(json.load(open(f)))
per_gpu = []
all_imgs = []
for r in workers:
    imgs = [i for i in r["images"] if i.get("counted")]
    all_imgs += imgs
    steady = [i["seconds"] for i in imgs if not i.get("first_image")]
    def stats(sel):
        s = [i["seconds"] for i in imgs if sel(i) and not i.get("first_image")]
        return {"n": len(s), "mean_s": round(statistics.mean(s), 2) if s else None}
    tot = sum(i["seconds"] for i in imgs)
    per_gpu.append({
        "worker": r["worker"], "gpu": r["gpu"], "images": len(imgs), "final_dtype": r.get("final_dtype"),
        "load": r["load"], "events": r["events"], "done": r.get("done", False),
        "peak_vram_allocated_gb": r.get("peak_vram_gb"), "peak_vram_reserved_gb": r.get("peak_reserved_gb"),
        "sec_per_image_all": round(tot / len(imgs), 2) if imgs else None,
        "sec_per_image_steady": round(statistics.mean(steady), 2) if steady else None,
        "images_per_hour_steady": round(3600 / statistics.mean(steady), 1) if steady else None,
        "text_only": stats(lambda i: not i.get("ref")), "with_reference": stats(lambda i: i.get("ref")),
        "first_image_s": next((i["seconds"] for i in imgs if i.get("first_image")), None),
    })
failures_first_try = sum(1 for i in all_imgs if i["attempts"][0]["bad"])
failures_final = sum(1 for i in all_imgs if i["bad"])
results = {
    "gpus": GPU_INFO, "n_gpu": N_GPU, "model": spec["model"], "resolution": f"{spec['width']}x{spec['height']}", "steps": spec["steps"],
    "download_s": DOWNLOAD_S, "wall_generation_s": WALL_S, "images_generated": len(all_imgs),
    "images_per_hour_per_gpu": [g["images_per_hour_steady"] for g in per_gpu],
    "images_per_hour_combined_steady": round(sum(g["images_per_hour_steady"] or 0 for g in per_gpu), 1),
    "images_per_hour_combined_wall_incl_loads": round(len(all_imgs) / WALL_S * 3600, 1) if WALL_S else None,
    "black_or_nan_first_try": failures_first_try, "black_or_nan_final": failures_final,
    "peak_vram_gb": max([g["peak_vram_allocated_gb"] or 0 for g in per_gpu] or [0]),
    "per_gpu": per_gpu,
}
json.dump(results, open(f"{OUT}/results.json", "w"), indent=1)
print(json.dumps({k: v for k, v in results.items() if k != "per_gpu"}, indent=1))
for g in per_gpu:
    print(json.dumps(g, indent=1))

In [ ]:
# contact sheet (self-contained HTML, white background, thumbnails embedded) + zip
import base64, html, io
from PIL import Image
byfile = {i["file"]: i for r in workers for i in r["images"]}
def thumb(fn):
    p = f"{OUT}/{fn}"
    if not os.path.exists(p): return None
    im = Image.open(p).convert("RGB"); im.thumbnail((300, 300))
    b = io.BytesIO(); im.save(b, "JPEG", quality=88)
    return base64.b64encode(b.getvalue()).decode()
def cell(fn):
    t = thumb(fn)
    info = byfile.get(fn)
    if t is None: return "<div class=c><div class=ph>not run</div></div>"
    cap = "copy of b panel 1" if info and info.get("copied_from") else (f"{info['seconds']:.2f}s {info['dtype']}" + (" ref" if info.get('ref') else "") + (" BAD" if info['bad'] else "") if info else "")
    return f"<div class=c><img src='data:image/jpeg;base64,{t}'><div class=t>{html.escape(fn)}<br>{cap}</div></div>"
rows = []
for c in spec["concepts"]:
    a = "".join(cell(f"{c['id']}_a_{p}.png") for p in range(1, 5)) if c["id"] in mode_a else "<div class=c><div class=ph>mode a not run for this concept</div></div>"
    b = "".join(cell(f"{c['id']}_b_{p}.png") for p in range(1, 5))
    rows.append(f"<h3>{html.escape(c['title'])}</h3><div class=row><div class=blk><h4>(a) text only, fixed seed {c['seed']}</h4><div class=ims>{a}</div></div><div class=blk><h4>(b) panels 2-4 conditioned on panel 1</h4><div class=ims>{b}</div></div></div>")
rows.append("<h3>Single-image style tests</h3><div class=ims>" + "".join(cell(f"{s['id']}_single_1.png") for s in spec["style_tests"]) + "</div>")
page = ("<!doctype html><meta charset=utf-8><title>Panel pilot contact sheet</title><style>body{background:#fff;color:#111;font:14px system-ui;margin:24px}"
        "h3{margin:28px 0 6px}h4{margin:0 0 4px;font-weight:600;color:#444}.row{display:flex;gap:28px;flex-wrap:wrap}.ims{display:flex;gap:6px}"
        ".c{width:150px}.c img{width:150px;height:150px;border:1px solid #ddd}.t{font-size:10px;color:#555}.ph{width:150px;height:150px;border:1px dashed #bbb;color:#999;display:flex;align-items:center;justify-content:center;text-align:center;font-size:11px}</style>"
        "<h2>FLUX.2 klein 4B pilot, 512x512, 4 steps</h2>" + "".join(rows))
open(f"{OUT}/contact_sheet.html", "w").write(page)
import shutil
shutil.copy("/kaggle/working/worker_0.log", f"{OUT}/worker_0.log")
if n_workers > 1: shutil.copy("/kaggle/working/worker_1.log", f"{OUT}/worker_1.log")
shutil.make_archive("/kaggle/working/panels", "zip", OUT)
print(os.listdir(OUT)[:100], len(os.listdir(OUT)))

In [ ]:
# Leave no stray processes behind.
print(subprocess.run("ps aux | grep -c '[w]orker.py'", shell=True, capture_output=True, text=True).stdout.strip(), "worker.py processes still running")